In [ ]:
import json
import re
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt

CSV_BID_FILE_SIZE = 6678514674        # bytes
CSV_PERSON_FILE_SIZE = 880941057      # bytes
JSON_BID_FILE_SIZE = 15878100724      # bytes
JSON_PERSON_FILE_SIZE = 1300941057    # bytes

def shorten_query(q):
    q = re.sub(r"\s+INTO\s+\w+", "", q, flags=re.IGNORECASE)
    q = re.sub(r'VARSIZED\("([^"]+)"\)', r'"\1"', q)
    q = re.sub(r'INT32\((\d+)\)', r'\1', q)
    q = re.sub(r'FLOAT64\(([\d.]+)\)', r'\1', q)
    return re.sub(r'\s+', ' ', q).strip()

def get_file_size(query, suite_name):
    is_json = "json" in suite_name.lower()
    if "bid" in query.lower():
        return JSON_BID_FILE_SIZE if is_json else CSV_BID_FILE_SIZE
    return JSON_PERSON_FILE_SIZE if is_json else CSV_PERSON_FILE_SIZE

def load_suite(filepath, suite_name):
    with open(filepath) as f:
        data = json.load(f)
    rows = []
    for r in data:
        if r.get("status") != "Stopped" or not r.get("started") or not r.get("stopped"):
            continue
        started = datetime.strptime(r["started"], "%Y-%m-%d %H:%M:%S.%f")
        stopped = datetime.strptime(r["stopped"], "%Y-%m-%d %H:%M:%S.%f")
        exec_time = (stopped - started).total_seconds()
        query = shorten_query(r["query"].strip())
        file_size = get_file_size(query, suite_name)
        rows.append({
            "suite": suite_name,
            "query": query,
            "threads": r["threads_number"],
            "exec_time_s": exec_time,
            "throughput_MBs": file_size / exec_time / (1024 * 1024),
        })
    return rows

rows = []
rows += load_suite("results/baseline_results_1776697527.json", "CSV base")
rows += load_suite("results/optimized_results_1776701104.json", "CSV lazy")
rows += load_suite("results/baseline-json_results_1776704138.json", "JSON base")
rows += load_suite("results/optimized-json_results_1776708719.json", "JSON lazy")

# # aggregation
# rows += load_suite("results/baseline_results_agg.json", "CSV base")
# rows += load_suite("results/optimized_results_agg.json", "CSV lazy")
# rows += load_suite("results/baseline-json_results_agg.json", "JSON base")

df = pd.DataFrame(rows)
print(f"Records: {len(df)}")
print(f"Suites: {sorted(df['suite'].unique())}")
print(f"Threads: {sorted(df['threads'].unique())}")
print(f"Queries: {sorted(df['query'].unique())}")
df

In [ ]:
# Grouped bar chart: 4 suites per thread count, one subplot per query
import numpy as np
import matplotlib.patheffects as pe

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica", "Liberation Sans"],
    "font.size": 11,
    "axes.edgecolor": "#333333",
    "axes.linewidth": 0.8,
})

queries = sorted(df["query"].unique())
suites = ["CSV base", "CSV lazy", "JSON base", "JSON lazy"]
suite_colors = {
    "CSV base":  "#0072B2",
    "CSV lazy": "#009E73",
    "JSON base": "#D55E00",
    "JSON lazy": "#CC79A7",
}
thread_counts = sorted(df["threads"].unique())
n_suites = len(suites)
bar_width = 0.18

n_queries = len(queries)
n_cols = min(3, n_queries)
n_rows = (n_queries + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 6 * n_rows),
                          sharey=False, squeeze=False)

x = np.arange(len(thread_counts))

for idx, q in enumerate(queries):
    row, col = divmod(idx, n_cols)
    ax = axes[row, col]
    for j, suite in enumerate(suites):
        s_data = df[(df["query"] == q) & (df["suite"] == suite)]
        vals = s_data.groupby("threads")["throughput_MBs"].median().reindex(thread_counts)
        offset = (j - (n_suites - 1) / 2) * bar_width
        bars = ax.bar(x + offset, vals.values, bar_width, label=suite,
                       color=suite_colors[suite], edgecolor="white", linewidth=0.5,
                       zorder=3)
        for bar in bars:
            h = bar.get_height()
            if h is not None and not np.isnan(h):
                txt = ax.text(bar.get_x() + bar.get_width() / 2, h + 1,
                              f"{h:.0f}", ha="center", va="bottom", fontsize=6,
                              color="#333333", fontweight="medium")
                txt.set_path_effects([pe.withStroke(linewidth=2, foreground="white")])

    ax.set_xlabel("Threads", fontsize=11, fontweight="medium")
    ax.set_ylabel("Throughput (MB/s)" if col == 0 else "", fontsize=11, fontweight="medium")
    ax.set_title(q, fontsize=10, fontweight="bold", pad=10)
    ax.set_xticks(x)
    ax.set_xticklabels(thread_counts)
    ax.set_ylim(0, ax.get_ylim()[1] * 1.15)
    ax.tick_params(axis="both", which="major", labelsize=10)
    ax.grid(True, alpha=0.25, axis="y", linestyle="--", zorder=0)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# Hide unused subplots
for idx in range(n_queries, n_rows * n_cols):
    row, col = divmod(idx, n_cols)
    axes[row, col].set_visible(False)

# Single shared legend at the bottom
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=n_suites, fontsize=10,
           framealpha=0.9, edgecolor="#cccccc", bbox_to_anchor=(0.5, -0.02))

fig.suptitle("CSV & JSON \u2014 Baseline vs. Optimized Throughput (lazy-parsing)",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Grouped bar chart aggregated by selectivity: one subplot per selectivity level
import numpy as np
import matplotlib.patheffects as pe

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica", "Liberation Sans"],
    "font.size": 11,
    "axes.edgecolor": "#333333",
    "axes.linewidth": 0.8,
})

SELECTIVITY_LABELS = {
    "id == 1018": "high",
    "id < 403348": "medium",
    "id < 2002507": "low",
    "id < 3800582": "very low",
    "bidder == 1001": "high",
    "auctionId < 2999100": "medium",
    "auctionId < 5990282": "low",
    "auctionId < 11401100": "very low",
}

def selectivity_label(q):
    for pat, lab in SELECTIVITY_LABELS.items():
        if pat in q:
            return lab
    return ""

df_sel = df.assign(selectivity=df["query"].map(selectivity_label))
df_sel = df_sel[df_sel["selectivity"] != ""]

selectivity_order = ["high", "medium", "low", "very low"]
suites = ["CSV base", "CSV lazy", "JSON base", "JSON lazy"]
suite_colors = {
    "CSV base":  "#0072B2",
    "CSV lazy": "#009E73",
    "JSON base": "#D55E00",
    "JSON lazy": "#CC79A7",
}
suite_hatch = {
    "CSV base":  "",
    "CSV lazy": "",
    "JSON base": "//",
    "JSON lazy": "//",
}
thread_counts = sorted(df_sel["threads"].unique())
n_suites = len(suites)
bar_width = 0.18
x = np.arange(len(thread_counts))

n_sel = len(selectivity_order)
n_cols = min(2, n_sel)
n_rows = (n_sel + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(8 * n_cols, 5.5 * n_rows),
                          sharey=False, squeeze=False)

# First take per-query median, then median across queries at that selectivity
agg = (
    df_sel.groupby(["selectivity", "suite", "threads", "query"])["throughput_MBs"]
    .median()
    .groupby(level=["selectivity", "suite", "threads"])
    .median()
)

for idx, sel in enumerate(selectivity_order):
    row, col = divmod(idx, n_cols)
    ax = axes[row, col]
    for j, suite in enumerate(suites):
        try:
            vals = agg.loc[(sel, suite)].reindex(thread_counts)
        except KeyError:
            continue
        offset = (j - (n_suites - 1) / 2) * bar_width
        bars = ax.bar(x + offset, vals.values, bar_width, label=suite,
                       color=suite_colors[suite], edgecolor="white", linewidth=0.5,
                       hatch=suite_hatch[suite], zorder=3)
        for bar in bars:
            h = bar.get_height()
            if h is not None and not np.isnan(h):
                txt = ax.text(bar.get_x() + bar.get_width() / 2, h + 1,
                              f"{h:.0f}", ha="center", va="bottom", fontsize=7,
                              color="#333333", fontweight="medium")
                txt.set_path_effects([pe.withStroke(linewidth=2, foreground="white")])

    n_queries_sel = df_sel[df_sel["selectivity"] == sel]["query"].nunique()
    ax.set_xlabel("Threads", fontsize=11, fontweight="medium")
    ax.set_ylabel("Throughput (MB/s)" if col == 0 else "", fontsize=11, fontweight="medium")
    ax.set_title(f"{sel} selectivity  (median over {n_queries_sel} queries)",
                 fontsize=11, fontweight="bold", pad=10)
    ax.set_xticks(x)
    ax.set_xticklabels(thread_counts)
    ax.set_ylim(0, ax.get_ylim()[1] * 1.15)
    ax.tick_params(axis="both", which="major", labelsize=10)
    ax.grid(True, alpha=0.25, axis="y", linestyle="--", zorder=0)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

for idx in range(n_sel, n_rows * n_cols):
    row, col = divmod(idx, n_cols)
    axes[row, col].set_visible(False)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=n_suites, fontsize=10,
           framealpha=0.9, edgecolor="#cccccc", bbox_to_anchor=(0.5, -0.02))

fig.suptitle("Throughput by Selectivity \u2014 Baseline vs. Optimized (lazy-parsing)",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Thesis-grade standalone plots per selectivity level.
# One PDF per level under plots/, no titles (captions go in LaTeX).
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

PLOTS_DIR = "plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

# Publication-quality style: sans-serif, moderate sizes, vector output.
thesis_rc = {
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica", "Liberation Sans"],
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 11,
    "axes.linewidth": 0.6,
    "axes.edgecolor": "#222222",
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "legend.frameon": False,
    "pdf.fonttype": 42,  # TrueType, embeds properly in LaTeX
    "ps.fonttype": 42,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
}

SELECTIVITY_LABELS = {
    "id == 1018": "very high",
    "id < 403348": "high",
    "id < 2002507": "medium",
    "id < 3800582": "low",
    "bidder == 1001": "very high",
    "auctionId < 2999100": "high",
    "auctionId < 5990282": "medium",
    "auctionId < 11401100": "low",
}

SELECTIVITY_FILE_SUFFIXES = {
    "very high": "single",
    "high": "25",
    "medium": "50",
    "low": "95"
}

def selectivity_label(q):
    for pat, lab in SELECTIVITY_LABELS.items():
        if pat in q:
            return lab
    return ""

df_sel = df.assign(selectivity=df["query"].map(selectivity_label))
df_sel = df_sel[df_sel["selectivity"] != ""]

selectivity_order = ["very high", "high", "medium", "low"]
suites = ["CSV base", "CSV lazy", "JSON base", "JSON lazy"]
# suite_colors = {
#     "CSV base":  "#0072B2",
#     "CSV lazy": "#009E73",
#     "JSON base": "#D55E00",
#     "JSON lazy": "#CC79A7",
# }
suite_colors = {
    "CSV base":  "#0072B2",
    "CSV lazy": "#009E73",
    "JSON base": "#D55E00",
    "JSON lazy": "#CC79A7",
}

suite_hatch = {
    "CSV base":  "",
    "CSV lazy": "//",
    "JSON base": "",
    "JSON lazy": "//",
}

thread_counts = sorted(df_sel["threads"].unique())
n_suites = len(suites)
bar_width = 0.18
x = np.arange(len(thread_counts))

agg = (
    df_sel.groupby(["selectivity", "suite", "threads", "query"])["throughput_MBs"]
    .median()
    .groupby(level=["selectivity", "suite", "threads"])
    .median()
)

with mpl.rc_context(thesis_rc):
    for sel in selectivity_order:
        fig, ax = plt.subplots(figsize=(5.5, 3.3), dpi=300)

        for j, suite in enumerate(suites):
            try:
                vals = agg.loc[(sel, suite)].reindex(thread_counts)
            except KeyError:
                continue
            offset = (j - (n_suites - 1) / 2) * bar_width
            bars = ax.bar(x + offset, vals.values, bar_width, label=suite,
                          color=suite_colors[suite], edgecolor="black",
                          linewidth=0.4, hatch=suite_hatch[suite], zorder=3)
            for bar in bars:
                h = bar.get_height()
                if h is not None and not np.isnan(h):
                    txt = ax.text(bar.get_x() + bar.get_width() / 2, h,
                                  f"{h:.0f}", ha="center", va="bottom",
                                  fontsize=4, color="#222222")
                    txt.set_path_effects([pe.withStroke(linewidth=1.8, foreground="white")])

        ax.set_xlabel("Threads")
        ax.set_ylabel("Throughput (MB/s)")
        ax.set_xticks(x)
        ax.set_xticklabels(thread_counts)
        ax.set_ylim(0, ax.get_ylim()[1] * 1.12)
        ax.grid(True, axis="y", linestyle=":", linewidth=0.4, alpha=0.6, zorder=0)
        ax.set_axisbelow(True)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.legend(loc="upper left", ncol=2, columnspacing=1.0, handlelength=1.4,
                  handletextpad=0.5, borderaxespad=0.3)

        fname = f"selectivity_{sel.replace(' ', '_')}_{SELECTIVITY_FILE_SUFFIXES[sel]}.pdf"
        out = os.path.join(PLOTS_DIR, fname)
        fig.savefig(out)
        print(f"wrote {out}")
        plt.show()


In [ ]:
# Median throughput across all queries per suite & thread count
import numpy as np
import matplotlib.patheffects as pe

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica", "Liberation Sans"],
    "font.size": 11,
    "axes.edgecolor": "#333333",
    "axes.linewidth": 0.8,
})

suites = ["CSV base", "CSV lazy", "JSON base", "JSON lazy"]
suite_colors = {
    "CSV base":  "#0072B2",
    "CSV lazy": "#009E73",
    "JSON base": "#D55E00",
    "JSON lazy": "#CC79A7",
}
thread_counts = sorted(df["threads"].unique())
n_suites = len(suites)
bar_width = 0.18
x = np.arange(len(thread_counts))

# For each (suite, threads) pair, first get per-query median, then median across queries
median_across = (
    df.groupby(["suite", "threads", "query"])["throughput_MBs"]
    .median()
    .groupby(level=["suite", "threads"])
    .median()
)

fig, ax = plt.subplots(figsize=(10, 6))

for j, suite in enumerate(suites):
    vals = median_across.loc[suite].reindex(thread_counts)
    offset = (j - (n_suites - 1) / 2) * bar_width
    bars = ax.bar(x + offset, vals.values, bar_width, label=suite,
                  color=suite_colors[suite], edgecolor="white", linewidth=0.5, zorder=3)
    for bar in bars:
        h = bar.get_height()
        if h is not None and not np.isnan(h):
            txt = ax.text(bar.get_x() + bar.get_width() / 2, h + 1,
                          f"{h:.0f}", ha="center", va="bottom", fontsize=8,
                          color="#333333", fontweight="medium")
            txt.set_path_effects([pe.withStroke(linewidth=2, foreground="white")])

ax.set_xlabel("Threads", fontsize=11, fontweight="medium")
ax.set_ylabel("Throughput (MB/s)", fontsize=11, fontweight="medium")
ax.set_title("Median Throughput Across All Queries — Baseline vs. Optimized (lazy-parsing)",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xticks(x)
ax.set_xticklabels(thread_counts)
ax.set_ylim(0, ax.get_ylim()[1] * 1.15)
ax.tick_params(axis="both", which="major", labelsize=10)
ax.grid(True, alpha=0.25, axis="y", linestyle="--", zorder=0)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(fontsize=10, framealpha=0.9, edgecolor="#cccccc")

fig.tight_layout()
plt.show()


In [ ]:
# Summary table: throughput (MB/s) all 4 suites
pivot = df.pivot_table(index=["query", "threads"], columns="suite", values="throughput_MBs", aggfunc="median")
pivot = pivot[["CSV base", "CSV lazy", "JSON base", "JSON lazy"]]
pivot["CSV speedup"] = pivot["CSV lazy"] / pivot["CSV base"]
pivot["JSON speedup"] = pivot["JSON lazy"] / pivot["JSON base"]
pivot.round(2)